# 06 — FAO GAEZ v5 Crop Suitability

**Owner:** Mostafa Elsherif
**Source:** FAO/IIASA Global Agro-Ecological Zones version 5
**Resolution:** ~1 km (0.00833 degrees), matches our 1 km grid
**Climate:** AgERA5 historical, period 2001-2020
**Setting:** Irrigated, high input level of management
**Output:** `data/processed/gaez_static.csv` — 560 rows, key = `cell_id`

## Why we need this

All our other datasets tell us what the land **is doing**: how green it is, how hot it gets, how much rain falls. None of them tell us what a crop **needs**.

GAEZ fills that gap. For each crop it answers three questions for every piece of land:

1. **Is this land suitable for this crop?** (suitability index 0-100)
2. **How much can it produce here?** (attainable yield in kg/ha)
3. **Which suitability class is it in?** (1 = best)

Without this our tool could say "the land is bare in May" but could not say "plant maize, not sugar beet".

## Which GAEZ product we use, and why

GAEZ v5 has many versions of the same data. The choice matters a lot:

| Mapset | What it is | Why we chose or rejected it |
|---|---|---|
| `RES05-SXX30AS` | Suitability index, continuous 0-10000, **all land** | Used. Continuous values are better for ranking crops. |
| `RES05-SCX30AS` | Suitability class, **current cropland only** | Used. Non-farm land is `nodata`, not a misleading zero. |
| `RES05-YCX30AS` | Attainable yield, **current cropland only** | Used. Gives kg/ha, which a farmer understands. |
| `RES05-SXX` | Same as the first, but ~10 km | Rejected. Too coarse for a 1 km grid. |
| `RES05-SIX30AS` | Class index, all land | Rejected. The cropland version is better for us. |

We chose **irrigated + high input** because our pilot area is fully irrigated Nile Delta farmland. Choosing rainfed would describe land that does not exist here.

## The 8 crops

| Code | Crop | Why it is in the list |
|---|---|---|
| WHE | Wheat | Egypt's main winter crop |
| MZE | Maize | Main summer crop |
| RCW | Wetland rice | Grown under flood irrigation in the north Delta |
| COT | Cotton | Egypt's historic export crop |
| ALF | Alfalfa | Stands for forage crops (berseem) |
| SUB | Sugar beet | A winter alternative to wheat |
| WPO | White potato | High value winter/spring crop |
| SOY | Soybean | A legume that puts nitrogen back in the soil |

A good rotation needs a winter crop, a summer crop, and a legume. This list covers all three.

## How we get the data without downloading 200 MB per crop

Each GAEZ file is a **global** raster of about 200 MB. We need a 22 x 29 km window. Downloading 24 files would be about 5 GB.

The files are **Cloud Optimized GeoTIFFs** stored on Google Cloud. That format lets a reader ask for only the bytes it needs over HTTP. `rasterio` does this for us: we open the URL, ask for our window, and only a few kilobytes come down.

## Plan

1. Open each GAEZ file by URL (no download).
2. Read only the window covering our pilot area, with a small padding.
3. Convert each 1 km cell's corners from UTM to lat/lon exactly, using `pyproj`.
4. For every cell, take the **mean of all GAEZ pixels inside that cell**.
5. Save one row per cell, 3 columns per crop.

## How to check the result is correct

- 560 rows, one per cell
- Wheat suitability should be **high** - this area is Egypt's wheat belt
- Wheat attainable yield should be roughly **4,000-8,000 kg/ha**
- Missing values should be few, and we must be able to explain them

In [5]:
%pip install rasterio pyproj -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import numpy as np, pandas as pd, rasterio
from rasterio.windows import from_bounds
from rasterio.transform import rowcol
from pyproj import Transformer

os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"

from pathlib import Path
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent          # works for everyone, not just one laptop
GRID = pd.read_csv(os.path.join(REPO, "data", "processed", "grid_1km.csv"))

W, S, E, N = 30.70, 31.05, 30.91, 31.30
PAD = 0.05
assert 31 <= S < N < 32 and 30 <= W < E < 31, "lat/lon swapped"

# exact cell corners: UTM 36N -> lat/lon
t = Transformer.from_crs("EPSG:32636", "EPSG:4326", always_xy=True)
lo0, la0 = t.transform(GRID.x_min_m.values, GRID.y_min_m.values)   # lower-left
lo1, la1 = t.transform(GRID.x_max_m.values, GRID.y_max_m.values)   # upper-right

CROPS = {"WHE": "wheat", "MZE": "maize", "RCW": "rice", "COT": "cotton",
         "ALF": "alfalfa", "SUB": "sugarbeet", "WPO": "potato", "SOY": "soybean"}

URL = ("https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/"
       "MAPSET/{ms}/GAEZ-V5.{ms}.HP0120.AGERA5.HIST.{c}.HILM.tif")


def cell_means(ms, crop, zero_is_nodata):
    """Mean of every GAEZ pixel whose area falls inside each 1 km grid cell.

    A single pixel at the cell centre is NOT enough: GAEZ pixels are ~0.93 km
    and our cells are 1 km, so the two grids do not line up. A cell centre can
    land on a canal or a village pixel while the rest of the cell is farmland.
    """
    with rasterio.open(URL.format(ms=ms, c=crop)) as src:
        win = from_bounds(W - PAD, S - PAD, E + PAD, N + PAD, src.transform)
        a   = src.read(1, window=win).astype("float64")
        tf  = src.window_transform(win)
        nod = src.nodata

    if nod is not None:
        a[a == nod] = np.nan
    if zero_is_nodata:
        a[a == 0] = np.nan
    a[a < 0] = np.nan

    r_top, c_left  = rowcol(tf, lo0, la1)
    r_bot, c_right = rowcol(tf, lo1, la0)
    r_top, c_left  = np.asarray(r_top),  np.asarray(c_left)
    r_bot, c_right = np.asarray(r_bot),  np.asarray(c_right)

    vals = np.full(len(GRID), np.nan)
    H, Wd = a.shape
    for i in range(len(GRID)):
        r0, r1 = max(0, min(r_top[i], r_bot[i])), min(H,  max(r_top[i], r_bot[i]) + 1)
        c0, c1 = max(0, min(c_left[i], c_right[i])), min(Wd, max(c_left[i], c_right[i]) + 1)
        if r1 > r0 and c1 > c0:
            block = a[r0:r1, c0:c1]
            if np.isfinite(block).any():
                vals[i] = np.nanmean(block)
    return vals


out = GRID[["cell_id"]].copy()

for code, name in CROPS.items():
    si  = cell_means("RES05-SXX30AS", code, zero_is_nodata=True)    # 0-10000
    cls = cell_means("RES05-SCX30AS", code, zero_is_nodata=True)    # class, 0 = nodata
    yld = cell_means("RES05-YCX30AS", code, zero_is_nodata=False)   # kg DW/ha

    out[f"si_{name}"]    = si / 100.0     # 0-10000 -> 0-100
    out[f"class_{name}"] = cls
    out[f"yield_{name}"] = yld

    print(f"{name:10s} si={np.nanmean(si)/100:6.2f}  class={np.nanmean(cls):5.2f}  "
          f"yield={np.nanmean(yld):8.0f} kg/ha  si_missing={int(np.isnan(si).sum())}")

OUT = os.path.join(REPO, "data", "processed", "gaez_static.csv")
out.to_csv(OUT, index=False)
print("\nsaved:", OUT, "| rows:", len(out), "| cols:", out.shape[1])

wheat      si= 85.53  class= 2.28  yield=    4820 kg/ha  si_missing=40
maize      si= 68.85  class= 3.14  yield=   10589 kg/ha  si_missing=40
rice       si= 64.42  class= 3.44  yield=    6167 kg/ha  si_missing=40
cotton     si= 64.41  class= 2.95  yield=    1024 kg/ha  si_missing=0
alfalfa    si= 66.84  class= 3.15  yield=   15521 kg/ha  si_missing=40
sugarbeet  si= 29.42  class= 5.17  yield=    3347 kg/ha  si_missing=0
potato     si= 59.54  class= 3.65  yield=    7431 kg/ha  si_missing=40
soybean    si= 57.22  class= 3.54  yield=    2975 kg/ha  si_missing=40

saved: C:\Field-Shift-Adapting-Farms-with-NASA-Data-2026\data\processed\gaez_static.csv | rows: 560 | cols: 25


In [7]:
# Cross-check the missing cells against our own Sentinel-2 farmland map
s2 = pd.read_csv(os.path.join(REPO, "data", "processed", "sentinel2_monthly.csv"))
farm = s2.groupby("cell_id").farm_pct.mean().rename("farm_pct")
chk = out.join(farm, on="cell_id")

m = chk.si_wheat.isna()
print("cells with no wheat suitability :", int(m.sum()), "of", len(chk))
print("mean farm_pct  (missing) :", round(chk.loc[m,  "farm_pct"].mean(), 3))
print("mean farm_pct  (present) :", round(chk.loc[~m, "farm_pct"].mean(), 3))
print("\nwheat suitability, cells with data: mean =",
      round(chk.loc[~m, "si_wheat"].mean(), 2))
print("\ncrop ranking by suitability:")
print(out[[c for c in out.columns if c.startswith("si_")]]
      .mean().sort_values(ascending=False).round(2))

cells with no wheat suitability : 40 of 560
mean farm_pct  (missing) : 0.935
mean farm_pct  (present) : 0.913

wheat suitability, cells with data: mean = 85.53

crop ranking by suitability:
si_wheat        85.53
si_maize        68.85
si_alfalfa      66.84
si_rice         64.42
si_cotton       64.41
si_potato       59.54
si_soybean      57.22
si_sugarbeet    29.42
dtype: float64


## Conclusion

We extracted 24 GAEZ layers (8 crops x 3 variables) at 1 km for all 560 cells. Total data moved over the network: a few hundred kilobytes instead of about 5 GB.

### Results

| Crop | Suitability (0-100) | Class (1 = best) | Attainable yield (kg DW/ha) |
|---|---|---|---|
| Wheat | 85.5 | 2.28 | 4,820 |
| Maize | 68.9 | 3.14 | 10,589 |
| Alfalfa | 66.8 | 3.15 | 15,521 |
| Rice | 64.4 | 3.44 | 6,167 |
| Cotton | 64.4 | 2.95 | 1,024 |
| Potato | 59.5 | 3.65 | 7,431 |
| Soybean | 57.2 | 3.54 | 2,975 |
| Sugar beet | 29.4 | 5.17 | 3,347 |

**Wheat comes first**, which is what we expect: this is Egypt's wheat belt. Alfalfa has the highest yield number because it is a forage crop cut several times a year, so its total biomass is large. Cotton has the lowest yield number because the harvested product is lint, which is a small part of the plant.

### A mistake we made, and how we found it

Our first attempt gave **68 cells with a wheat suitability of exactly 0**. In Egypt's wheat belt, that is not believable.

Our first guess was that those cells were water or buildings. We tested that guess against our **own** Sentinel-2 farmland map:

| | mean `farm_pct` |
|---|---|
| cells with suitability 0 | 0.86 |
| cells with suitability > 0 | 0.92 |

A cell that is 86% farmland is not a river and not a town. **The guess was wrong.**

The real cause was our method, not the data. We were reading **one pixel at the centre of each cell**. GAEZ pixels are about 0.93 km and our cells are 1 km, so the two grids do not line up. A cell centre sometimes landed on a single canal or village pixel while the whole square around it was farmland.

We fixed it two ways at once:

1. Take the **mean of every GAEZ pixel inside the cell**, not one pixel.
2. Use the **current cropland** versions of the data, where non-farm land is marked `nodata` instead of being given a suitability of zero.

Missing cells dropped from **68 to 40** (7.1%). We leave those 40 as empty values rather than filling them with a guessed number, and the merged table records them honestly as missing.

This is why we cross-check every dataset against another one. A number that looks like data can still be an artefact of how we read it.

### One result we do not accept yet

GAEZ rates **sugar beet lowest** of all eight crops (29.4, class 5.17). This needs checking against what is actually grown in our pilot area before we let our tool discourage a crop. If farmers here do grow sugar beet successfully, then we have found a real gap between the FAO model and the ground, and we will report that gap instead of hiding it.

### What this unlocks

`gaez_static.csv` gives our tool, for every 1 km cell, the suitability and expected yield of 8 candidate crops. Combined with the cover gaps from Sentinel-2 and the irrigation need from SMAP, we can now score a rotation rather than just describe the land.